# Logistic Regression and Classification

In our previous notebook, we explored **Linear Regression** to predict continuous values like taxi fares. The output of a linear regression model can be any real number (from negative infinity to positive infinity).

But what if we want to predict a category instead of a continuous number? This is called **Classification**.
* **Binary Classification**: Predicting one of two possible categories (e.g., "Will it rain tomorrow?" -> Yes or No; "Is this email spam?" -> Spam or Not Spam).
* We typically represent the categories as `1` (Yes/Spam) and `0` (No/Not Spam).

### Why Linear Regression Fails for Classification
If we use our linear model equation $y = wx + b$ for classification, the model will draw a straight line.
1. The line can predict values like `1.5` or `-0.5`. What does a prediction of `1.5` mean when the only valid options are `0` or `1`? It makes no sense!
2. We need a model that outputs a **probability** (a value strictly between `0` and `1`) so we can interpret it as "there is an 80% chance of rain tomorrow".

### Entering Logistic Regression
This is where **Logistic Regression** comes in. Despite its name, Logistic Regression is actually a **classification** algorithm, not a regression algorithm.

It helps us by predicting the **probability** that a given input belongs to a specific category (e.g., a 75% probability that it will rain).

To do this, it takes our linear regression equation $z = wx + b$ (which can output any value) and maps it into a probability between 0 and 1 using the **Sigmoid Function**.

### Entering the Sigmoid Function
To solve this, we take the output of our linear equation (let's call it $z$):
$$z = w_1x_1 + w_2x_2 + ... + b$$

And we pass it through a mathematical function called the **Sigmoid Function** (also known as the Logistic Function), represented by $\sigma(z)$:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This function has an S-shaped curve that perfectly "squishes" any input value $z$ into a value between `0` and `1`:
* **Extreme Positive Input ($z \to \infty$)**: $e^{-z}$ becomes almost $0$, so $\sigma(z) \approx \frac{1}{1 + 0} = 1$.
* **Extreme Negative Input ($z \to -\infty$)**: $e^{-z}$ becomes extremely large, making the denominator huge. So $\sigma(z) \approx \frac{1}{\infty} = 0$.
* **Zero Input ($z = 0$)**: $e^0 = 1$, so $\sigma(0) = \frac{1}{1 + 1} = 0.5$ (exactly a 50% chance).


In [ ]:
# @title Coding Example: How Sigmoid Helps Us Categorize
import numpy as np

# Let's define the Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Problem: Will a student pass the exam based on hours studied?
# Features (x): Hours studied
# Let's assume we already trained a model and found these parameters:
weight = 0.8  # weight (w)
bias = -3.0   # bias (b)

# Let's test for different study hours
study_hours = np.array([1, 3, 4, 6, 8])

# Step 1: Calculate the linear output (z)
z_values = (weight * study_hours) + bias

# Step 2: Pass through Sigmoid to get probabilities (y_predicted)
probabilities = sigmoid(z_values)

# Step 3: Categorize (Thresholding at 0.5: >= 0.5 is Pass (1), else Fail (0))
predictions = (probabilities >= 0.5).astype(int)

# Print the results clearly
print(f"{'Hours Studied':<15} | {'Linear Output (z)':<18} | {'Probability (y\')':<18} | {'Final Prediction':<15}")
print("-" * 75)
for h, z, prob, pred in zip(study_hours, z_values, probabilities, predictions):
    status = "Pass (1)" if pred == 1 else "Fail (0)"
    print(f"{h:<15} | {z:<18.2f} | {prob:<18.4f} | {status:<15}")


Hours Studied   | Linear Output (z)  | Probability (y')   | Final Prediction
---------------------------------------------------------------------------
1               | -2.20              | 0.0998             | Fail (0)       
3               | -0.60              | 0.3543             | Fail (0)       
4               | 0.20               | 0.5498             | Pass (1)       
6               | 1.80               | 0.8581             | Pass (1)       
8               | 3.40               | 0.9677             | Pass (1)       


## 📉 2. Key Concepts: Log Loss vs. Mean Squared Error (MSE)

When training our model, we need a way to calculate **how wrong** the model's guesses are. We call this **Loss**.

In **Linear Regression**, we used **Mean Squared Error (MSE)**, which calculates the average squared difference between the actual taxi fare and the predicted fare.

However, for **Logistic Regression**, we use **Log Loss** instead. Here is why MSE doesn't work well here in a simple way:

 MSE makes learning too slow when the model is wildly wrong:
*   Because Logistic Regression uses the Sigmoid function to force outputs between `0` and `1`, the curve becomes extremely flat at the very ends (near `0` and `1`).
*   If the model is highly confident but completely wrong (e.g., it predicts a `99%` chance of rain, but it turns out to be a sunny day), MSE's mathematical curve becomes so flat that the model barely learns from its mistake. It gets "stuck" and updates its weights at a snail's pace.
*   **Log Loss** fixes this by using a logarithmic scale. If the model is highly confident but wrong, Log Loss shoots up to infinity, giving the model a massive "nudge" to quickly correct itself.



## 🛡️ 3. Regularization and the "Dog on a Leash" Analogy

Overfitting happens when a model memorizes details of the training data instead of learning general patterns. To prevent this, we use a technique called **Regularization** (specifically **L2 Regularization** or Ridge).

To understand this simply, let's use an analogy:

### 🐕 The Dog on a Leash Analogy

Imagine you are training a puppy (our model) to stay on a sidewalk (the decision boundary between Pass/Fail or 0/1):

1.  **Runaway Weights (The Wild Dog without a Leash)**:
    *   If the training data is **perfectly separable** (e.g., all students who studied $<3$ hours failed, and all who studied $\ge 4$ hours passed), the puppy discovers that the further it runs away from the sidewalk, the more "correct" its predictions are.
    *   The Sigmoid function wants to predict exactly `100%` or `0%`. But Sigmoid can only do this if the input is **infinity**.
    *   Without a leash, the model will continuously inflate the weight ($w$) during training, trying to reach infinity. The weight "runs away"!
    *   **Why this is bad**: The model becomes extremely overconfident. If a new student studies 3.9 hours, the model might panic and make a wildly wrong prediction because its weights are unnaturally huge.

2.  **Regularization (The Leash)**:
    *   Regularization acts like a **leash** on the dog.
    *   It tells the model: "You want to be correct, but if you run too far (if your weights grow too large), you will receive a penalty score."
    *   This penalty forces the model to keep its weights small and sensible, making it much more robust and prepared for new, slightly different data.


In [6]:
# @title Coding Example: Regularization in Action (Leash vs. No Leash)
from sklearn.linear_model import LogisticRegression
import numpy as np

# Perfectly linearly separable dataset
# Hours studied: 1, 2 (fail, y=0) vs 5, 6 (pass, y=1)
X = np.array([1, 2, 5, 6]).reshape(-1, 1)
y = np.array([0, 0, 1, 1])

# Model A: NO LEASH (Almost no regularization, C is huge)
model_no_leash = LogisticRegression(C=1e6, solver='lbfgs')
model_no_leash.fit(X, y)

# Model B: STRONG LEASH (Strong regularization, C is small)
model_strong_leash = LogisticRegression(C=0.1, solver='lbfgs')
model_strong_leash.fit(X, y)

# Print comparison
print(f"{'Model Type':<25} | {'Learned Weight (w)':<20} | {'Base Fee / Bias (b)':<20}")
print("-" * 75)
print(f"{'Model A (No Leash)':<25} | {model_no_leash.coef_[0][0]:<20.4f} | {model_no_leash.intercept_[0]:<20.4f}")
print(f"{'Model B (Strong Leash)':<25} | {model_strong_leash.coef_[0][0]:<20.4f} | {model_strong_leash.intercept_[0]:<20.4f}")

print("\nObservations:")
print("- Model A (No Leash) has an extremely large weight, making its decision boundary highly rigid and overconfident.")
print("- Model B (Strong Leash) keeps the weight small and sensible, making it much safer for predicting on new data!")


Model Type                | Learned Weight (w)   | Base Fee / Bias (b) 
---------------------------------------------------------------------------
Model A (No Leash)        | 6.3032               | -21.1475            
Model B (Strong Leash)    | 0.2836               | -0.9927             

Observations:
- Model A (No Leash) has an extremely large weight, making its decision boundary highly rigid and overconfident.
- Model B (Strong Leash) keeps the weight small and sensible, making it much safer for predicting on new data!


## 🔄 4. Training Loss, Validation Loss, and Early Stopping

When we train a machine learning model, we want it to perform well on **new, unseen data**, not just memorize the examples we show it. To check this, we split our data into two main sets:

### 📚 The Exam Analogy

Imagine a student preparing for a history exam:
1.  **Training Loss (Practice Questions)**:
    *   These are the practice questions the student studies over and over again.
    *   **Training Loss** measures the errors on these study questions. As the student spends more time studying, their score on these questions will naturally get better and better (Training Loss goes down).
2.  **Validation Loss (Mock Test)**:
    *   This is a fresh mock test with brand-new questions the student has never seen before.
    *   **Validation Loss** measures the errors on this mock test. This tells us if the student actually *understands* history, or if they have just *memorized* the answers to the practice questions.

---

### 📈 Overfitting and Early Stopping

*   **Early Phase**: At the start of studying, both Training Loss and Validation Loss go down. The student is learning general rules.
*   **Overfitting Phase**: Eventually, if the student studies *too* much, they will start memorizing the exact punctuation and typos of the practice questions. Their score on the practice questions will be perfect (Training Loss continues to drop to near $0$), but their score on the mock test will get **worse** (Validation Loss starts to rise). This is called **Overfitting**.

### How Early Stopping Helps
**Early Stopping** is a smart safety switch. During training, we monitor the Validation Loss after each study round (epoch). The moment the Validation Loss stops decreasing and begins to rise, we **stop training immediately**.

This ensures we keep the model at its peak performance on new data, before it has a chance to overfit and memorize the training set.


## 📊 5. Making Decisions: Classification Threshold & Confusion Matrix

Once our Logistic Regression model outputs a probability (e.g., $0.75$ or a $75\%$ chance of rain), how do we make a final decision: "Will it rain?" (Yes or No)?

We do this by setting a **Classification Threshold**.

### 1. The Threshold (The Decision Line)
By default, the threshold is **$0.5$ (50%)**:
*   If Probability $\ge 0.5 \to$ We predict **Yes** (Rain).
*   If Probability $< 0.5 \to$ We predict **No** (Sunny).

However, we can change this threshold!
*   If we are planning an outdoor wedding, we might want to be extremely safe. We could set the threshold to **$0.2$ (20%)**—so if there is even a tiny $20\%$ chance of rain, we predict "Yes" and move indoors.
*   If we are deciding whether to cancel school for a snowstorm, we might want to be very sure. We could set the threshold to **$0.8$ (80%)** so we only predict "Yes" if we are highly confident.

---

### 2. The Confusion Matrix (The Scorecard)
After making predictions, how do we know if we got them right? We use a scorecard called a **Confusion Matrix**. It is a $2 \times 2$ table that compares our guesses against reality.

Imagine we made predictions for 10 days:

| | Actual: Rain (1) | Actual: Sunny (0) |
| :--- | :--- | :--- |
| **Predicted: Rain (1)** | **True Positive (TP)** <br> We predicted Rain, and it rained! (Success) | **False Positive (FP)** <br> We predicted Rain, but it was Sunny. (False Alarm) |
| **Predicted: Sunny (0)** | **False Negative (FN)** <br> We predicted Sunny, but it rained. (Missed) | **True Negative (TN)** <br> We predicted Sunny, and it was Sunny! (Success) |


In [ ]:
# @title Coding Example: Scorecard (Confusion Matrix) at 0.5 vs. 0.7 Threshold
import numpy as np

# Let's simulate 10 actual days of weather observations:
# 1 = Rain, 0 = Sunny
y_actual = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])

# The model's predicted probabilities of rain for those 10 days:
y_predicted_probs = np.array([0.80, 0.10, 0.60, 0.95, 0.40, 0.20, 0.75, 0.30, 0.55, 0.45])

def calculate_confusion_matrix(probs, actual, threshold):
    # Apply the threshold slider
    predictions = (probs >= threshold).astype(int)

    # Count results
    tp = np.sum((predictions == 1) & (actual == 1))
    fp = np.sum((predictions == 1) & (actual == 0))
    fn = np.sum((predictions == 0) & (actual == 1))
    tn = np.sum((predictions == 0) & (actual == 0))
    return tp, fp, fn, tn

for threshold in [0.5, 0.7]:
    tp, fp, fn, tn = calculate_confusion_matrix(y_predicted_probs, y_actual, threshold)
    print(f"\n=== Scorecard (Confusion Matrix) at Threshold = {threshold} ===")
    print(f"{'':<20} | {'Actual: Rain (1)':<18} | {'Actual: Sunny (0)':<18}")
    print("-" * 65)
    print(f"{'Predicted: Rain (1)':<20} | {f'True Positive: {tp}':<18} | {f'False Positive: {fp}':<18}")
    print(f"{'Predicted: Sunny (0)':<20} | {f'False Negative: {fn}':<18} | {f'True Negative: {tn}':<18}")


## 📈 6. Evaluating the Model: Classification Metrics

Once we have our Confusion Matrix scorecard (True Positives, False Positives, False Negatives, True Negatives), we can calculate simple scores to evaluate our model. 

Here are the four main metrics explained in a very simple way:

### 1. Accuracy (The General Score)
*   **What it means**: Out of all our predictions, what percentage did we get right?
*   **Analogy**: Your score on a school test (e.g., 8 out of 10 correct = 80% accuracy).
*   **Formula**: `(True Positives + True Negatives) / Total Predictions`

### 2. Precision (Avoiding False Alarms)
*   **What it means**: When we predict it **will** rain, how often are we actually right?
*   **Why it matters**: A high precision means we have **very few False Alarms** (False Positives). If we predict rain, you can trust it and bring an umbrella.
*   **Formula**: `True Positives / (True Positives + False Positives)`

### 3. Recall / Sensitivity (Avoiding Missed Events)
*   **What it means**: Out of all the days it **actually** rained, how many did we successfully catch?
*   **Why it matters**: A high recall means we have **very few Missed Events** (False Negatives). If it rains, we almost certainly predicted it.
*   **Formula**: `True Positives / (True Positives + False Negatives)`

### 4. F1-Score (The Balance)
*   **What it means**: A single balanced score that combines both Precision and Recall. If one is very bad, the F1-score drops significantly.
*   **Formula**: `2 * (Precision * Recall) / (Precision + Recall)`


In [ ]:
# @title Coding Example: Calculating Metrics for Threshold 0.5 vs. 0.7
# We reuse the TP, FP, FN, TN values calculated in the previous step:

def calculate_metrics(tp, fp, fn, tn):
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return accuracy, precision, recall, f1

print(f"{'Threshold':<10} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("-" * 62)
for threshold in [0.5, 0.7]:
    tp, fp, fn, tn = calculate_confusion_matrix(y_predicted_probs, y_actual, threshold)
    acc, prec, rec, f1 = calculate_metrics(tp, fp, fn, tn)
    print(f"{threshold:<10} | {acc:<10.2%} | {prec:<10.2%} | {rec:<10.2%} | {f1:<10.2%}")

print("\nObservation:")
print("- When we increased the threshold to 0.7, we became more cautious. Our Precision went up (no false alarms!), but our Recall went down (we missed some actual rain days!).")


## 📉 7. The Overall Score: ROC Curve and AUC

How do we evaluate our model's performance across **all possible thresholds** (from 0 to 1) at once, instead of picking just one? We use the **ROC Curve** and **AUC**.

### 1. The ROC Curve (The Trade-Off Slider)
Imagine a slider that lets you choose any threshold from $0.0$ to $1.0$:
*   If we slide it to $0.0$, we predict "Rain" every single day. We will catch all rainy days (100% Recall), but we will also have endless False Alarms.
*   If we slide it to $1.0$, we never predict "Rain". We will have zero False Alarms, but we will miss all actual rain days.

The **ROC Curve** is a simple graph that plots this trade-off. It shows how our True Positive Rate (Recall) improves against our False Positive Rate (False Alarm rate) as we slide the threshold.

---

### 2. The AUC (The School Grade)
**AUC** (Area Under the ROC Curve) is a single number between $0.5$ and $1.0$ that measures the entire area under that curve. Think of it as the model's overall grade:
*   **AUC = 1.0 (A+)**: Perfect model. It separates Rain days from Sunny days flawlessly.
*   **AUC = 0.8 - 0.9 (B to A)**: Good to Excellent.
*   **AUC = 0.5 (F)**: Guessing. The model is as good as flipping a coin.


In [ ]:
# @title Coding Example: Plotting the ROC Curve & Calculating AUC
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Calculate the ROC curve points
fpr, tpr, thresholds = roc_curve(y_actual, y_predicted_probs)

# Calculate the AUC score
auc_score = roc_auc_score(y_actual, y_predicted_probs)

# Plot the ROC Curve
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color="teal", lw=2, label=f"Model ROC Curve (AUC = {auc_score:.2f})")
plt.plot([0, 1], [0, 1], color="orange", linestyle="--", label="Coin Flip Guessing (AUC = 0.50)")
plt.xlabel("False Alarm Rate (False Positive Rate)")
plt.ylabel("Recall Rate (True Positive Rate)")
plt.title("ROC Curve: Measuring Overall Model Quality")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


## ⚖️ 8. Prediction Bias

Ideally, a model's average predictions should match the average reality. 

**Prediction Bias** measures this difference. It is a simple score:
$$\text{Prediction Bias} = \text{Average of Predicted Probabilities} - \text{Average of Actual Labels}$$

*   **Bias = 0**: Perfect. On average, our predictions match reality.
*   **Positive Bias (Over-predicting)**: The model predicts rain too often. For example, if the model's average predicted probability is $0.60$ (60%) but it only actually rained on $40\%$ of the days, the model has a positive bias of $+0.20$.
*   **Negative Bias (Under-predicting)**: The model is too pessimistic and predicts rain less often than it actually occurs.

A high prediction bias indicates that there is a systematic issue in the model or the dataset that needs to be corrected during training.


In [ ]:
# @title Coding Example: Calculating Prediction Bias
import numpy as np

# Calculate average predicted probability
avg_predicted = np.mean(y_predicted_probs)

# Calculate average actual rate
avg_actual = np.mean(y_actual)

# Bias is the difference
bias = avg_predicted - avg_actual

print(f"Average Predicted Probability of Rain: {avg_predicted:.2%}")
print(f"Actual Rain Rate in Dataset:           {avg_actual:.2%}")
print(f"Prediction Bias:                       {bias:+.2%}")

if abs(bias) < 0.05:
    print("\nConclusion: The model has very low bias. On average, its guesses match reality!")
elif bias > 0:
    print("\nConclusion: The model has positive bias. It tends to over-predict rain.")
else:
    print("\nConclusion: The model has negative bias. It tends to under-predict rain.")


## 🌈 9. Multi-Class Classification

So far, we have only predicted **Binary** outcomes: Yes or No, $0$ or $1$. 

But what if we want to predict between **three or more categories**? This is called **Multi-Class Classification**.
*   *Example*: Predicting tomorrow's weather category: **Sunny**, **Rainy**, or **Snowy**.

### How Logistic Regression Handles Multiple Classes
A standard Logistic Regression model can only handle two classes. To handle three or more, it uses one of two strategies:

1.  **One-vs-All (OVA)**:
    *   We train a separate binary model for each class.
    *   Model 1: Sunny vs. Not Sunny
    *   Model 2: Rainy vs. Not Rainy
    *   Model 3: Snowy vs. Not Snowy
    *   During prediction, we run all three models and pick the one with the highest probability.
2.  **Softmax Regression (Multinomial)**:
    *   A single unified model that calculates probabilities for all classes simultaneously, ensuring they all add up to exactly $1.0$ (100%).


In [ ]:
# @title Coding Example: Multi-Class Predictions
import numpy as np

# Imagine our Softmax model predicts probabilities for 3 classes:
# Index 0 = Sunny, Index 1 = Rainy, Index 2 = Snowy
categories = ["Sunny", "Rainy", "Snowy"]

# Simulated predicted probabilities for a single day:
predicted_probabilities = np.array([0.05, 0.80, 0.15])  # Sums to 1.0

# The model picks the category with the HIGHEST probability
predicted_class_index = np.argmax(predicted_probabilities)
predicted_category = categories[predicted_class_index]

print("Model Predicted Probabilities:")
for cat, prob in zip(categories, predicted_probabilities):
    print(f"- {cat:<8}: {prob:.2%}")

print(f"\nFinal Predicted Category: {predicted_category} (Index {predicted_class_index})")


### Libraries Used in this Project

*   **Pandas:** Think of this as "Excel for Python." It helps us organize and clean data using tables called DataFrames.
*   **NumPy:** Used for fast mathematical calculations and handling large lists of numbers (arrays).
*   **Scikit-Learn (sklearn):** The primary toolbox for Machine Learning. It contains the algorithms and tools needed to train and test models.
    *   `train_test_split`: Helps us hide some data from the model so we can test it later.
    *   `LogisticRegression`: The "brain" that learns the patterns in the weather data.
    *   `accuracy_score`: A simple way to see what percentage of our predictions were correct.

### What are we doing in this exercise?

In this exercise, we are performing **Binary Classification**.

**The Goal:** We want to look at today's weather patterns (like Temperature, Humidity, and Pressure) and predict a "Yes" or "No" answer to the question: **"Will it rain tomorrow?"**

We use **Logistic Regression** because it is a great algorithm for predicting categories (like Yes/No). It looks at historical data, finds correlations between weather factors and rainfall, and creates a mathematical boundary to classify new weather data.

## Australian Rainfall Prediction

**Objective:** Build a classification model using Logistic Regression to predict whether it will rain tomorrow in Australia based on weather observations.

## Step 1: Download the Dataset

The dataset is sourced from Kaggle ('Rain in Australia'). Since we are running in **Google Colab**, we download it directly from our GitHub repository using a URL — no manual upload needed!

> **Note:** The full dataset (~145,000 rows) is available on [Kaggle](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package). If you have a Kaggle account, you can upload your `kaggle.json` API key to Colab and use the Kaggle API to download it directly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -------------------------------------------------------------------
# Load data directly from GitHub (works in Colab without any upload!)
# The dataset is the 'Rain in Australia' dataset from Kaggle.
# -------------------------------------------------------------------
DATA_URL = 'https://raw.githubusercontent.com/akshaykhanna/learning-complete-ai/main/weather.csv'

df = pd.read_csv(DATA_URL)

# Print shape and first 5 rows
print("Data Shape:", df.shape)
print(df.head())

Data Shape: (145460, 24)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity3pm  Pressure9am  \
0           W           44.0          W  ...        22.0       1007.7   
1         WNW           44.0        NNW  ...        25.0       1010.6   
2         WSW           46.0          W  ...        30.0       1007.6   
3          NE           24.0         SE  ...        16.0       1017.6   
4           W           41.0        ENE  ...        33.0       1010.8   

   Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  RainTomo

# Data Preprocessing

In this step, we need to clean the dataset by dropping redundant columns that do not contribute to the prediction or may cause data leakage. We also need to remove any null values to ensure the model can be trained effectively.

In [ ]:
# Drop redundant columns
df = df.drop(['Date', 'Location', 'RISK_MM'], axis=1)

# Remove null values
df = df.dropna()

print("Cleaned Data Shape:", df.shape)

Cleaned Data Shape: (56420, 21)


# Splitting Data & Building the Model

To build the model, we first isolate the target variable (`RainTomorrow`) from the features. We then perform a 75/25 train-test split to ensure we have a separate set of data to evaluate the model's performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Convert categorical variables to dummy/indicator variables
df = pd.get_dummies(df, drop_first=True)

# Isolate target variable
X = df.drop('RainTomorrow_Yes', axis=1)
y = df['RainTomorrow_Yes']

# Train-Test Split (75/25)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize and fit the model
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=5000)

# Evaluation

After training the model, we test its accuracy on the unseen test data to see how well it generalizes to new observations.

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 85.66%
